# Classification à partir de mesures cytologiques

Ce notebook accompagne l'exercice 5.5 et sert de synthèse du chapitre. Le jeu *Wisconsin Diagnostic Breast Cancer* contient $569$ observations décrites par $30$ mesures réelles extraites d'images cytologiques.

Nous comparerons un SVM affine, un SVM gaussien et un MLP. L'objectif est d'étudier un jeu de données et les choix mathématiques nécessaires à cette étude ; il ne s'agit pas de valider un dispositif de diagnostic médical.

## Parcours

1. [Données, étiquettes et fréquences](#donnees-wisconsin)
2. [Partition et centrage-réduction](#pretraitement-wisconsin)
3. [Projection sur deux composantes principales](#pca-wisconsin)
4. [SVM affine et SVM gaussien](#svm-wisconsin)
5. [MLP avec Flax NNX](#mlp-wisconsin)
6. [Erreurs et marges signées](#erreurs-wisconsin)
7. [Erreurs asymétriques](#poids-wisconsin)

In [1]:
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
import flax
from flax import nnx
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

print(
    f"JAX {jax.__version__}, Flax {flax.__version__}, "
    f"Optax {optax.__version__}, scikit-learn {sklearn.__version__}"
)

JAX 0.11.1, Flax 0.12.9, Optax 0.2.8, scikit-learn 1.9.0


<a id="donnees-wisconsin"></a>
## 1. Données, étiquettes et fréquences

Charger le jeu avec `load_breast_cancer`. Dans `scikit-learn`, la cible `0` désigne une tumeur maligne et la cible `1` une tumeur bénigne. Adopter la convention du cours
$$
z=+1\quad\text{maligne},\qquad z=-1\quad\text{bénigne}.
$$

Vérifier les dimensions, afficher les noms des premières mesures et calculer les fréquences des deux classes.

In [ ]:
# À compléter.

<a id="pretraitement-wisconsin"></a>
## 2. Partition et centrage-réduction

Construire une partition fixée et stratifiée ('stratified split') comportant environ $60\%$ de données d'apprentissage, $20\%$ de validation et $20\%$ de test.

Comparer les ordres de grandeur des mesures, puis ajuster `StandardScaler` **uniquement** sur les données d'apprentissage. Utiliser ensuite la même transformation pour les deux autres ensembles. Employer les données de validation ou de test lors de ce calcul constituerait une fuite d'information ('data leakage').

In [ ]:
# À compléter.

<a id="pca-wisconsin"></a>
## 3. Projection sur deux composantes principales

Calculer les deux premières composantes principales à partir des seules données d'apprentissage centrées et réduites. Projeter les trois ensembles sur le plan obtenu.

Une séparation dans ce plan fournirait un score affine séparant aussi les données dans $\mathbb R^{30}$. En revanche, une superposition dans le plan projeté ne permet pas de conclure que les données ne sont pas séparables dans l'espace initial.

In [ ]:
# À compléter.

<a id="svm-wisconsin"></a>
## 4. SVM affine et SVM gaussien

Nous utilisons la classe `SVC` de `scikit-learn`, dont l'implémentation repose sur la bibliothèque LIBSVM.

Pour chaque machine, choisir les hyperparamètres sur l'ensemble de validation :

- $C$ pour le SVM affine ;
- $C$ et $\gamma$ pour le SVM gaussien.

Le paramètre `C` de `SVC` est inversement lié à la force de la régularisation. En cas d'égalité des fréquences de validation, utiliser la perte charnière moyenne comme second critère.

In [5]:
def perte_charniere(scores, z):
    return np.mean(np.maximum(0.0, 1.0 - z * scores))

def selectionner_svm(noyau, valeurs_C, valeurs_gamma=(None,)):
    candidats = []
    for C in valeurs_C:
        for gamma in valeurs_gamma:
            arguments = {"kernel": noyau, "C": C}
            if gamma is not None:
                arguments["gamma"] = gamma
            modele = SVC(**arguments)
            modele.fit(x_app_n, z_app)
            scores = modele.decision_function(x_val_n)
            erreur = np.mean(np.sign(scores) != z_val)
            candidats.append((erreur, perte_charniere(scores, z_val), C, gamma, modele))
    return min(candidats, key=lambda resultat: (resultat[0], resultat[1]))

In [ ]:
# À compléter.

<a id="mlp-wisconsin"></a>
## 5. MLP avec Flax NNX

Ajuster un MLP à deux couches cachées avec la perte logistique régularisée. Comparer quelques largeurs, coefficients de régularisation et initialisations. Retenir la machine en utilisant la fréquence d'erreur puis la perte logistique sur l'ensemble de validation.

Les fonctions d'apprentissage sont fournies ; l'exercice porte sur la construction de la comparaison et son interprétation.

In [7]:
class MLPBinaire(nnx.Module):
    def __init__(self, dimension, largeur, *, rngs):
        self.affine_1 = nnx.Linear(dimension, largeur, rngs=rngs)
        self.affine_2 = nnx.Linear(largeur, largeur, rngs=rngs)
        self.affine_3 = nnx.Linear(largeur, 1, rngs=rngs)

    def __call__(self, x):
        y = jnp.tanh(self.affine_1(x))
        y = jnp.tanh(self.affine_2(y))
        return self.affine_3(y)[..., 0]

def norme_parametres(machine):
    return sum(
        jnp.sum(feuille**2)
        for feuille in jax.tree.leaves(nnx.state(machine, nnx.Param))
    )

def cout_mlp(machine, x, z, lamb):
    return (
        jnp.mean(jax.nn.softplus(-z * machine(x)))
        + 0.5 * lamb * norme_parametres(machine)
    )

@nnx.jit
def pas_mlp(machine, optimiseur, x, z, lamb):
    fonction = lambda m: cout_mlp(m, x, z, lamb)
    valeur, gradient = nnx.value_and_grad(fonction)(machine)
    optimiseur.update(machine, gradient)
    return valeur

def entrainer_mlp(x, z, largeur, lamb, graine, *, iterations=1200):
    machine = MLPBinaire(x.shape[1], largeur, rngs=nnx.Rngs(graine))
    optimiseur = nnx.Optimizer(machine, optax.adam(5.0e-3), wrt=nnx.Param)
    x_jax = jnp.asarray(x)
    z_jax = jnp.asarray(z)
    for _ in range(iterations):
        valeur = pas_mlp(machine, optimiseur, x_jax, z_jax, lamb)
    return machine, float(valeur)

In [ ]:
# À compléter.

<a id="erreurs-wisconsin"></a>
## 6. Erreurs et marges signées

Comparer les trois machines sur les ensembles d'apprentissage, de validation et de test. Sur le test, distinguer :

- les faux négatifs ('false negatives') : tumeurs malignes classées comme bénignes ;
- les faux positifs ('false positives') : tumeurs bénignes classées comme malignes.

Représenter les marges signées. Leur signe est comparable entre machines, mais leur échelle dépend de la normalisation du score.

In [ ]:
# À compléter.

In [ ]:
# À compléter.

<a id="poids-wisconsin"></a>
## 7. Erreurs asymétriques

Dans une application, les deux types d'erreurs peuvent recevoir des importances différentes. Le paramètre `class_weight` de `SVC` permet de pondérer les termes de la perte charnière.

Ajuster des SVM affines avec les rapports $\omega_{+1}/\omega_{-1}=1,2,5$. Observer l'évolution des faux négatifs et des faux positifs sur les données de test. Ces poids ne sont pas déduits automatiquement des données : ils traduisent un choix propre à l'application.

In [ ]:
# À compléter.

## Bilan

Ce jeu réel réunit les principaux choix du chapitre : représentation des données, score, perte, régularisation, validation et mesure des erreurs. Le SVM à noyau et le MLP peuvent améliorer un score affine, mais leur comparaison n'est pertinente que si tout le protocole expérimental est commun.